# Smoke — costo de la Fase 2 del benchmark (RLE / máscaras)

**Pregunta que responde:** ¿vale la pena pivotar e incluir ya la **Fase 2**
(métricas de máscara) en el benchmark, o es demasiado caro frente a la deadline?

**Premisa corregida (importante):** en `tracking.py` las máscaras se computan
**siempre** (el detector produce `det.mask`; la caja sale de la máscara). El flag
`include_masks` **no** cambia el cómputo del modelo — solo decide si se **codifica a
RLE** y se embebe en el JSON. Por eso el costo de la Fase 2 NO está en la inferencia,
sino en: (a) codificar RLE + JSON más grande, y (b) el **post-pass** de métricas
(decodificar RLE por frame + IoU temporal).

Este notebook mide, sobre **1 video seeded del testing**:
1. tiempo de tracking **sin** RLE (`include_masks=False`),
2. tiempo de tracking **con** RLE (`include_masks=True`),
3. tamaño de los dos JSON,
4. tiempo de un **mini post-pass** de IoU temporal de máscara sobre el JSON con RLE.

Corre **en el pod** (necesita SAM3 + GPU). `render_video=False` para aislar el costo
de inferencia del de escribir el mp4.

In [ ]:
import time
from pathlib import Path

import pandas as pd
import torch

from src.data.metadata import _load_metadata_config
from src.utils import get_abs_path

# Selección reproducible: 5 videos al azar del split de testing (2), tomamos el 1ro.
SEED = 42
N_SAMPLE = 5

_, metadata_csv, _, _ = _load_metadata_config()
df = pd.read_csv(get_abs_path(metadata_csv))
testing = df[df["split"] == 2].sort_values("id")
seeded = testing.sample(n=min(N_SAMPLE, len(testing)), random_state=SEED)
print(f"videos seeded del testing (seed={SEED}):")
for _, r in seeded.iterrows():
    print(f"  id={r['id']}  {r['ruta']}")

VIDEO = str(seeded.iloc[0]["ruta"])
print(f"\nvideo de prueba: {VIDEO}")

In [ ]:
# Carga única de SAM3 (reusada en ambas corridas).
from src.core.sam3_loader import load_sam3
from src.core.inference import run_inference

bundle = load_sam3()
print("SAM3 cargado.")


def time_tracking(include_masks: bool, out_stem: str):
    """Corre tracking sobre VIDEO, devuelve (segundos, vram_pico_MB, ruta_json, bytes)."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    out = get_abs_path("outputs") / f"_smoke_{out_stem}.mp4"
    out.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    res = run_inference(
        VIDEO,
        mode="tracking",
        output_path=out,
        sampling="all",          # video completo (no acotar frames)
        include_masks=include_masks,
        render_video=False,      # aislar costo de inferencia
        bundle=bundle,
    )
    dt = time.perf_counter() - t0
    vram = (
        torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0.0
    )
    jp = Path(res["json"])
    return dt, vram, jp, jp.stat().st_size

In [ ]:
# 1) Sin RLE.
t_no, vram_no, json_no, size_no = time_tracking(include_masks=False, out_stem="norle")
print(f"[sin RLE]  {t_no:7.1f} s | VRAM pico {vram_no:8.1f} MB | JSON {size_no/1e6:7.2f} MB")

# 2) Con RLE.
t_yes, vram_yes, json_yes, size_yes = time_tracking(include_masks=True, out_stem="rle")
print(f"[con RLE]  {t_yes:7.1f} s | VRAM pico {vram_yes:8.1f} MB | JSON {size_yes/1e6:7.2f} MB")

print("\n== overhead marginal de include_masks ==")
print(f"  tiempo : x{t_yes / max(t_no, 1e-9):.2f}  (+{t_yes - t_no:.1f} s)")
print(f"  JSON   : x{size_yes / max(size_no, 1):.1f}  ({size_no/1e6:.2f} -> {size_yes/1e6:.2f} MB)")

In [ ]:
# 3) Mini post-pass Fase 2: IoU temporal de máscara entre frames consecutivos,
#    por obj_id y por clase, sobre el JSON con RLE. Mide cuánto cuesta este pase
#    (decodificar RLE + IoU), que es el costo REAL de la Fase 2 (no la inferencia).
import json

import numpy as np

from src.core.inference_schema import decode_rle


def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union else 1.0


t0 = time.perf_counter()
doc = json.loads(json_yes.read_text(encoding="utf-8"))
frames = doc.get("frames", [])

prev = {}  # (clase, obj_id) -> máscara del frame anterior
ious = []
for fr in frames:
    cur = {}
    for cls, dets in fr.get("detections", {}).items():
        for d in dets:
            if "rle" not in d:
                continue
            m = decode_rle(d["rle"])
            key = (cls, d["obj_id"])
            cur[key] = m
            if key in prev:
                ious.append(mask_iou(prev[key], m))
    prev = cur

dt_pp = time.perf_counter() - t0
print(f"post-pass IoU temporal: {dt_pp:.1f} s sobre {len(frames)} frames, {len(ious)} pares")
if ious:
    arr = np.array(ious)
    print(f"  IoU temporal medio={arr.mean():.3f}  mediana={np.median(arr):.3f}  min={arr.min():.3f}")

## Cómo leer el resultado / criterio de decisión

- **Si el overhead de `include_masks` en tiempo es chico** (p. ej. <1.2×) y el JSON,
  aunque más grande, es manejable → **pivotar e incluir la Fase 2 desde ya** sale
  barato: solo hay que correr los lotes con `include_masks=True` y añadir el post-pass
  de métricas de máscara.
- **Si el JSON con RLE explota** (×10–×50) y/o el post-pass es lento sobre un solo
  video → multiplicado por 5 videos × 6 configs puede no caber antes de la deadline;
  conviene **mantener la Fase 2 diferida**.

Nota: el grueso del tiempo (forward de SAM3/YOLO) es el mismo con o sin RLE; lo que
este smoke aísla es exactamente el **costo extra** que añade la Fase 2.